# Retail Orders — Executable Data Profile

Purpose: turn the raw retail-order request into measurable, repeatable checks for completeness, uniqueness, validity, consistency and freshness. The notebook reads `retail-orders-raw.csv` and produces a machine-checkable profile and KPI readiness result.

**Decision owner:** Commerce / Operations Manager.  
**KPI grain:** order line, except order-count and payment-rate denominators which use distinct valid orders.  
**Refresh expectation:** daily.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

DATA_PATH = 'retail-orders-raw.csv'  # upload/copy the CSV beside this notebook
TODAY = pd.Timestamp('2026-09-09')  # replace with pd.Timestamp.today().normalize() in production

raw = pd.read_csv(DATA_PATH)
print('Shape:', raw.shape)
print('Columns:', list(raw.columns))
raw.head()


In [ ]:
# Normalization used for categorical consistency checks

df = raw.copy()
for c in ['customer_segment','payment_status','category','city']:
    if c in df.columns:
        df[c+'_norm'] = df[c].astype('string').str.strip()

df['customer_segment_norm'] = df['customer_segment_norm'].str.title()
df['payment_status_norm'] = df['payment_status_norm'].str.title()
df['category_norm'] = df['category_norm'].str.replace(r'\s+', ' ', regex=True)

df['order_date_parsed'] = pd.to_datetime(df['order_date'], errors='coerce', dayfirst=False)
# Explicitly try DD/MM/YYYY where the first parse fails
mask = df['order_date_parsed'].isna() & df['order_date'].notna()
df.loc[mask, 'order_date_parsed'] = pd.to_datetime(df.loc[mask,'order_date'], errors='coerce', dayfirst=True)

print(df[['order_date','order_date_parsed','customer_segment_norm','payment_status_norm']].head())


In [ ]:
# Executable quality checks
allowed_segments = {'Student','Fresher','Professional'}
allowed_categories = {'Learning Kit','Course Access','Mentor Session'}
allowed_payments = {'Paid','Pending','Failed','Refunded'}

checks = []
def add_check(dimension, field, check, passed, observed, threshold, action):
    checks.append({
        'dimension':dimension,'field':field,'check':check,'passed':bool(passed),
        'observed':observed,'threshold':threshold,'failure_action':action
    })

# Completeness
add_check('Completeness','order_id','non-null',df['order_id'].notna().all(),f"{df['order_id'].notna().mean()*100:.1f}% non-null",'100%','Block publication')
add_check('Completeness','order_date','non-null',df['order_date'].notna().all(),f"{df['order_date'].notna().mean()*100:.1f}% non-null",'100%','Block publication')
add_check('Completeness','city','non-empty',df['city'].fillna('').astype(str).str.strip().ne('').mean()>=.99,f"{df['city'].fillna('').astype(str).str.strip().ne('').mean()*100:.1f}% non-empty",'>=99%','Warn; block if <95%')
add_check('Completeness','discount_pct','missing only with justification',df['discount_pct'].notna().all(),f"{df['discount_pct'].notna().mean()*100:.1f}% non-null",'100% unless justified','Block discount/net-sales KPI')

# Uniqueness
add_check('Uniqueness','order_id','duplicate count == 0',df['order_id'].dropna().is_unique,int(df['order_id'].duplicated().sum()),'0','Block publication')

# Validity
valid_dates = df['order_date_parsed'].notna() & df['order_date_parsed'].between(pd.Timestamp('2025-01-01'), TODAY)
add_check('Validity','order_date','parseable and within allowed range',valid_dates.all(),f"{valid_dates.mean()*100:.1f}% valid",'100%','Block publication')
add_check('Validity','customer_segment','allowed values after normalization',df['customer_segment_norm'].isin(allowed_segments).all(),sorted(set(df.loc[~df['customer_segment_norm'].isin(allowed_segments),'customer_segment_norm'].dropna().tolist())),'100%','Block affected slices')
add_check('Validity','category','allowed values',df['category_norm'].isin(allowed_categories).all(),sorted(set(df.loc[~df['category_norm'].isin(allowed_categories),'category_norm'].dropna().tolist())),'100%','Block affected slices')
qty_num = pd.to_numeric(df['quantity'], errors='coerce')
qty_valid = qty_num.notna() & (qty_num > 0) & (qty_num % 1 == 0)
add_check('Validity','quantity','whole number > 0',qty_valid.all(),f"{qty_valid.mean()*100:.1f}% valid",'100%','Block sales KPIs')
price_num = pd.to_numeric(df['unit_price'], errors='coerce')
price_valid = price_num.notna() & (price_num >= 0)
add_check('Validity','unit_price','numeric and >= 0',price_valid.all(),f"{price_valid.mean()*100:.1f}% valid",'100%','Block sales KPIs')
disc_num = pd.to_numeric(df['discount_pct'], errors='coerce')
disc_valid = disc_num.notna() & disc_num.between(0,100)
add_check('Validity','discount_pct','numeric in [0,100]',disc_valid.all(),f"{disc_valid.mean()*100:.1f}% valid",'100%','Block sales KPIs')
pay_valid = df['payment_status_norm'].isin(allowed_payments)
add_check('Validity','payment_status','allowed after normalization',pay_valid.all(),f"{pay_valid.mean()*100:.1f}% valid",'100%','Block payment KPIs')

# Consistency
add_check('Consistency','payment_status','normalization resolves case/whitespace variants',df['payment_status_norm'].notna().all(),sorted(df['payment_status_norm'].dropna().unique().tolist()),'100% normalized','Block payment KPIs')
valid_rows = qty_valid & price_valid & disc_valid
calc_gross = qty_num * price_num
calc_discount = calc_gross * disc_num / 100
calc_net = calc_gross - calc_discount
add_check('Consistency','commercial arithmetic','net = gross - discount for valid rows',np.isclose(calc_net[valid_rows], (calc_gross-calc_discount)[valid_rows]).all(),f"{valid_rows.sum()}/{len(df)} rows calculable",'100% of valid rows','Block sales KPIs')

# Freshness: business-date proxy because source has no ingestion timestamp
latest_date = df['order_date_parsed'].max()
age_days = (TODAY - latest_date).days if pd.notna(latest_date) else None
fresh_pass = age_days is not None and age_days <= 7
add_check('Freshness','dataset','latest business date <= 7 days from expected refresh date',fresh_pass,f"latest={latest_date.date() if pd.notna(latest_date) else None}; age={age_days} days",'Hard fail if >7 days','Escalate; label dashboard stale')

profile = pd.DataFrame(checks)
profile


In [ ]:
# KPI readiness and summary
summary = profile.groupby('dimension').agg(checks=('passed','size'), passed=('passed','sum'))
summary['pass_rate_pct'] = summary['passed'] / summary['checks'] * 100

print(summary)
print('\nOverall checks passed:', int(profile['passed'].sum()), '/', len(profile))
print('Overall contract status:', 'PASS' if profile['passed'].all() else 'FAIL')
print('\nFailed checks:')
print(profile.loc[~profile['passed'], ['dimension','field','check','observed','threshold','failure_action']].to_string(index=False))


In [ ]:
# KPI calculations only on contract-valid sales rows
valid_sales = valid_rows & df['order_id'].notna() & df['order_date_parsed'].notna() & df['category_norm'].isin(allowed_categories)

kpi = {
    'Gross Order Value': float(calc_gross[valid_sales].sum()),
    'Discount Amount': float(calc_discount[valid_sales].sum()),
    'Net Sales': float(calc_net[valid_sales].sum()),
    'Paid Revenue': float(calc_net[valid_sales & (df['payment_status_norm']=='Paid')].sum()),
    'Valid Order Count': int(df.loc[valid_sales, 'order_id'].nunique()),
    'Units Sold': float(qty_num[valid_sales].sum()),
}
kpi['Average Order Value'] = kpi['Net Sales'] / kpi['Valid Order Count'] if kpi['Valid Order Count'] else np.nan
valid_orders = df.loc[valid_sales, ['order_id','payment_status_norm']].drop_duplicates('order_id')
base = len(valid_orders)
kpi['Payment Success Rate'] = (valid_orders['payment_status_norm'].eq('Paid').sum()/base*100) if base else np.nan
kpi['Refund Rate'] = (valid_orders['payment_status_norm'].eq('Refunded').sum()/base*100) if base else np.nan
kpi['Pending Order Rate'] = (valid_orders['payment_status_norm'].eq('Pending').sum()/base*100) if base else np.nan

pd.Series(kpi, name='value')


## Interpretation for this sample

The sample intentionally contains data-quality defects (for example a duplicate order ID, an invalid date, a negative quantity, an out-of-range discount, a missing city/discount, and a text quantity). The notebook therefore **fails the contract rather than silently correcting the data**. Normalization is allowed for case/whitespace variants such as `paid` → `Paid`; business-changing corrections require an owner-approved rule.

Freshness uses `order_date` only as a business-date proxy because the raw file has no ingestion/last-updated timestamp. In production, add an ingestion timestamp and use that for the freshness SLA.
